# Day 5: Parameter Controls & Testing
Objective: Systematically test generation parameters and understand
how each one controls model behavior.
Build a reusable parameter testing function.

In [1]:
!pip install transformers torch -q

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.


## 1. Reusable Test Function
Instead of rewriting model.generate() every time,
wrap it in a function that accepts any parameter combination.

In [3]:
def test_generation(prompt, temperature=1.0, top_k=50, top_p=1.0,
                    max_new_tokens=60, do_sample=True,
                    num_beams=1, repetition_penalty=1.0, seed=42):

    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt")

    output = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        num_beams=num_beams,
        repetition_penalty=repetition_penalty
    )

    result = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Prompt     : {prompt}")
    print(f"Params     : temp={temperature}, top_k={top_k}, top_p={top_p}, rep_penalty={repetition_penalty}")
    print(f"Output     : {result}")
    print("-" * 60)
    return result

## 2. Temperature Tests
How does temperature alone change output quality?

In [4]:
prompt = "The future of AI research is"

print("=== TEMPERATURE SWEEP ===\n")
for temp in [0.3, 0.7, 1.0, 1.3, 1.8]:
    test_generation(prompt, temperature=temp, top_k=0, top_p=1.0)

=== TEMPERATURE SWEEP ===

Prompt     : The future of AI research is
Params     : temp=0.3, top_k=0, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is not just about AI, it is about the future of AI. It is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It is
------------------------------------------------------------
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=0, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is uncertain and could not be guaranteed in the future,” he said.
------------------------------------------------------------
Prompt     : The future of AI research is
Params     : temp=1.0, top_k=0, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is solid and intelligent work is required in this area,” he wrote.
---------------------------------------------

## 3. Top-K Tests
How does changing K affect vocabulary diversity?

In [5]:
print("=== TOP-K SWEEP ===\n")
for k in [1, 10, 50, 200, 0]:  # 0 = disabled
    label = "disabled" if k == 0 else str(k)
    print(f"top_k = {label}")
    test_generation(prompt, temperature=0.7, top_k=k, top_p=1.0)

=== TOP-K SWEEP ===

top_k = 1
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=1, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is a matter of time before AI is able to be used to solve problems.













































------------------------------------------------------------
top_k = 10
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=10, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is uncertain and could not be guaranteed in the future,” he said.
------------------------------------------------------------
top_k = 50
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=50, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is uncertain and could not be guaranteed in the future,” he said.
------------------------------------------------------------
top_k = 200
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=200, top_p=1.0

## 4. Top-P Tests
How does nucleus size affect output?

In [6]:
print("=== TOP-P SWEEP ===\n")
for p in [0.5, 0.7, 0.9, 0.95, 1.0]:
    test_generation(prompt, temperature=0.7, top_k=0, top_p=p)

=== TOP-P SWEEP ===

Prompt     : The future of AI research is
Params     : temp=0.7, top_k=0, top_p=0.5, rep_penalty=1.0
Output     : The future of AI research is not just about AI, it is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It is about the future of AI research. It
------------------------------------------------------------
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=0, top_p=0.7, rep_penalty=1.0
Output     : The future of AI research is uncertain, but the future of AI research is uncertain, but the future of AI research is uncertain, but the future of AI research is uncertain, but the future of AI research is uncertain, but the future of AI research is uncertain, but the future of AI research is uncertain, but the future of
------------------------------------------------------------
Prompt     : The futur

## 5. Repetition Penalty Tests
Does it actually fix the repetition problem we saw in Day 1?

In [7]:
print("=== REPETITION PENALTY ===\n")
for penalty in [1.0, 1.2, 1.5, 2.0]:
    test_generation(prompt, temperature=0.7, repetition_penalty=penalty)

=== REPETITION PENALTY ===

Prompt     : The future of AI research is
Params     : temp=0.7, top_k=50, top_p=1.0, rep_penalty=1.0
Output     : The future of AI research is uncertain and could not be guaranteed in the future,” he said.
------------------------------------------------------------
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=50, top_p=1.0, rep_penalty=1.2
Output     : The future of AI research is uncertain and could not be guaranteed in the near term.
“We want to bring these people together, get them involved with this new industry – for example what you can do when there are too many robots here on earth… There's a lot going on now that we have no idea how much
------------------------------------------------------------
Prompt     : The future of AI research is
Params     : temp=0.7, top_k=50, top_p=1.0, rep_penalty=1.5
Output     : The future of AI research is uncertain and could not be guaranteed in the near term.
“We want to bring these peop

## 6. Multi-Prompt Test
Same parameters, different prompts — how sensitive is output to prompt wording?

In [8]:
params = dict(temperature=0.7, top_k=50, top_p=0.9)
prompts = [
    "Machine learning engineers must",
    "The biggest challenge in AI is",
    "In 2030, most software will",
    "A good neural network should"
]

print("=== MULTI-PROMPT TEST (same params) ===\n")
for p in prompts:
    test_generation(p, **params)

=== MULTI-PROMPT TEST (same params) ===

Prompt     : Machine learning engineers must
Params     : temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.0
Output     : Machine learning engineers must understand and understand the complex ways in which the world is changing.















































------------------------------------------------------------
Prompt     : The biggest challenge in AI is
Params     : temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.0
Output     : The biggest challenge in AI is to make AI work better. We know that AI is a complex system and we know that we can learn from it. We know that AI is a complex system and we know that we can learn from it. We know that AI is a complex system and we know that we can learn from it.
------------------------------------------------------------
Prompt     : In 2030, most software will
Params     : temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.0
Output     : In 2030, most software will be used by a large proport

## 7. Best Parameters Summary
Based on all experiments, what combination works best?

In [9]:
print("=== BEST CONFIG TEST ===\n")
best_prompts = [
    "The future of AI research is",
    "Machine learning engineers must",
    "The biggest challenge in AI is"
]

for p in best_prompts:
    test_generation(
        p,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        max_new_tokens=80
    )

=== BEST CONFIG TEST ===

Prompt     : The future of AI research is
Params     : temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.2
Output     : The future of AI research is uncertain, but a new report in the journal Proceedings OF National Academy Press on Tuesday shows that we can learn from our mistakes.
"I think it's an important question for us to be able see what people are doing," says Thomas van Dijk and his colleagues at Princeton University. "There may even have been some differences between groups."
In other words: We might not know much about how
------------------------------------------------------------
Prompt     : Machine learning engineers must
Params     : temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.2
Output     : Machine learning engineers must understand and develop a new approach to their industry.
------------------------------------------------------------
Prompt     : The biggest challenge in AI is
Params     : temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.2
Output

## 8. Observations & Notes

### Temperature
- Low (0.3): focused but repetitive/collapses
- Medium (0.7): best balance — coherent and varied
- High (1.5+): creative but often incoherent

### Top-K
- K=1: always picks top token = deterministic (same as greedy)
- K=50: good balance
- K=0 (disabled): full vocabulary available, needs top_p to control

### Top-P
- Low (0.5): very conservative, few candidates
- High (0.95-1.0): nearly unrestricted
- 0.9 is the industry sweet spot

### Repetition Penalty
- 1.0 = no penalty
- 1.2 = mild, fixes loops without hurting quality
- 2.0+ = too aggressive, output gets unnatural

### Key Insight
No single parameter controls output quality alone.
temperature + top_k + top_p + repetition_penalty work together.
Industry standard: temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.2

### My Questions / Surprises
(write your own after running)